# Notebook 2 — Equal-NFE Solver Inversions

This notebook strengthens the paper's **equal-compute solver comparison** without re-running diffusion sampling.

It assumes the project already contains:

- `ddim_residual_data_512/`
  - `z.pt`
  - `teacher_1000.pt`
- `solver_generalization/`
  - saved solver outputs such as `ddim_50.pt`, `deis_50.pt`, `dpmpp_50.pt`, `unipc_50.pt`
  - and the corresponding `*_100.pt` files

The notebook:

1. loads the **actual saved solver outputs**;
2. computes held-out baseline MSE to the fixed DDIM-1000 reference;
3. trains the same small correction family used in the paper;
4. computes held-out residual NMSE;
5. detects **accuracy–difficulty inversions** automatically;
6. exports CSV, LaTeX, and a paper-ready figure.

No numerical result is hard-coded.

In [6]:
from pathlib import Path
import random
import copy
import json
import math

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt
# ============================================================
# GLOBAL CONFIG
# ============================================================

# Reproducibility
SEED = 20260927

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Project paths
PROJECT_DIR = Path(".")

DATA_DIR = PROJECT_DIR / "ddim_residual_data_512"
SOLVER_DIR = PROJECT_DIR / "solver_generalization"
OUT_DIR = PROJECT_DIR / "equal_nfe_validation"

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# DATA SPLIT
# Original dataset = 512 samples
# 384 train / 64 validation / 64 test
# ============================================================

N_TOTAL = 512

N_TRAIN = 384
N_VAL = 64
N_TEST = 64

TRAIN_END = N_TRAIN                 # 384
VAL_END = N_TRAIN + N_VAL          # 448
TEST_START = VAL_END                # 448

# ============================================================
# EXPERIMENT CONFIG
# ============================================================

NFES = [50, 100]

# For today's faster equal-NFE experiment
WIDTHS = [8]

SEEDS = [0, 1, 2]

# Corrector training
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MAX_EPOCHS = 1000
PATIENCE = 40
MIN_DELTA = 1e-5

# Solver aliases for file discovery
ALIASES = {
    "DDIM": ["ddim"],
    "DEIS": ["deis"],
    "DPM++": ["dpmpp", "dpm++", "dpm_solver", "dpmsolver"],
    "UniPC": ["unipc", "uni_pc"],
}

# ============================================================
# CHECK CONFIG
# ============================================================

print("========== CONFIG ==========")
print("SEED:", SEED)
print("device:", device)

print("\nPaths:")
print("DATA_DIR:", DATA_DIR.resolve())
print("SOLVER_DIR:", SOLVER_DIR.resolve())
print("OUT_DIR:", OUT_DIR.resolve())

print("\nSplit:")
print("N_TOTAL:", N_TOTAL)
print("N_TRAIN:", N_TRAIN)
print("N_VAL:", N_VAL)
print("N_TEST:", N_TEST)
print("TRAIN_END:", TRAIN_END)
print("VAL_END:", VAL_END)

print("\nExperiment:")
print("NFES:", NFES)
print("WIDTHS:", WIDTHS)
print("SEEDS:", SEEDS)

========== CONFIG ==========
SEED: 20260927
device: cpu

Paths:
DATA_DIR: D:\PhD EMP\Paper Writting\STEP_COMPLEXITY\ddim_residual_data_512
SOLVER_DIR: D:\PhD EMP\Paper Writting\STEP_COMPLEXITY\solver_generalization
OUT_DIR: D:\PhD EMP\Paper Writting\STEP_COMPLEXITY\equal_nfe_validation

Split:
N_TOTAL: 512
N_TRAIN: 384
N_VAL: 64
N_TEST: 64
TRAIN_END: 384
VAL_END: 448

Experiment:
NFES: [50, 100]
WIDTHS: [8]
SEEDS: [0, 1, 2]


## 1. Load the fixed reference and dataset split

In [7]:
z = torch.load(DATA_DIR / "z.pt", map_location="cpu")
teacher = torch.load(DATA_DIR / "teacher_1000.pt", map_location="cpu")

assert len(z) == 512
assert len(teacher) == 512

N_TRAIN = 384
N_VAL = 64
TRAIN_END = N_TRAIN
VAL_END = N_TRAIN + N_VAL

z_train = z[:TRAIN_END]
z_val = z[TRAIN_END:VAL_END]
z_test = z[VAL_END:]

teacher_train = teacher[:TRAIN_END]
teacher_val = teacher[TRAIN_END:VAL_END]
teacher_test = teacher[VAL_END:]

print("split:", len(z_train), len(z_val), len(z_test))

split: 384 64 64


## 2. Discover the saved solver outputs

We intentionally discover files from disk rather than hard-code results.

Expected naming convention is approximately:

- `ddim_50.pt`
- `deis_50.pt`
- `dpmpp_50.pt`
- `unipc_50.pt`

and the same for `100`.

If your exact filenames differ slightly, edit only `ALIASES`.

In [8]:
ALIASES = {
    "DDIM": ["ddim"],
    "DEIS": ["deis"],
    "DPM++": ["dpmpp", "dpm++", "dpm_solver", "dpmsolver"],
    "UniPC": ["unipc", "uni_pc"],
}

NFES = [50, 100]

all_pt = list(SOLVER_DIR.glob("*.pt"))
print("Found .pt files:")
for p in sorted(all_pt):
    print(" ", p.name)

def find_solver_file(solver_name, nfe):
    aliases = ALIASES[solver_name]
    candidates = []

    for p in all_pt:
        stem = p.stem.lower()
        if str(nfe) not in stem:
            continue
        if any(a.lower() in stem for a in aliases):
            candidates.append(p)

    if len(candidates) == 0:
        raise FileNotFoundError(
            f"No saved output found for {solver_name}, NFE={nfe} in {SOLVER_DIR}."
        )

    if len(candidates) > 1:
        # Prefer exact-looking names if available.
        candidates = sorted(
            candidates,
            key=lambda p: (len(p.stem), p.name)
        )

    return candidates[0]

solver_files = {}
for solver in ALIASES:
    for nfe in NFES:
        p = find_solver_file(solver, nfe)
        solver_files[(solver, nfe)] = p
        print(f"{solver:5s} NFE={nfe}: {p.name}")

Found .pt files:
  ddim_10.pt
  ddim_100.pt
  ddim_20.pt
  ddim_200.pt
  ddim_50.pt
  deis_10.pt
  deis_100.pt
  deis_20.pt
  deis_200.pt
  deis_50.pt
  dpmpp_10.pt
  dpmpp_100.pt
  dpmpp_20.pt
  dpmpp_200.pt
  dpmpp_50.pt
  dpmpp_reference_1000_128.pt
  unipc_10.pt
  unipc_100.pt
  unipc_20.pt
  unipc_200.pt
  unipc_50.pt
DDIM  NFE=50: ddim_50.pt
DDIM  NFE=100: ddim_100.pt
DEIS  NFE=50: deis_50.pt
DEIS  NFE=100: deis_100.pt
DPM++ NFE=50: dpmpp_50.pt
DPM++ NFE=100: dpmpp_100.pt
UniPC NFE=50: unipc_50.pt
UniPC NFE=100: unipc_100.pt


## 3. Load outputs and validate shapes

In [9]:
solver_outputs = {}

for key, path in solver_files.items():
    x = torch.load(path, map_location="cpu")

    if isinstance(x, dict):
        # Handle common saved-dict conventions without guessing silently.
        tensor_keys = [
            k for k, v in x.items()
            if torch.is_tensor(v) and v.ndim == 4
        ]
        if len(tensor_keys) != 1:
            raise ValueError(
                f"{path.name} is a dict with ambiguous tensor keys: {tensor_keys}"
            )
        x = x[tensor_keys[0]]

    if not torch.is_tensor(x):
        raise TypeError(f"{path.name} did not contain a tensor.")

    if len(x) != len(teacher):
        raise ValueError(
            f"{path.name}: expected {len(teacher)} samples, got {len(x)}."
        )

    solver_outputs[key] = x.float()
    print(key, x.shape, x.dtype)

('DDIM', 50) torch.Size([512, 3, 32, 32]) torch.float32
('DDIM', 100) torch.Size([512, 3, 32, 32]) torch.float32
('DEIS', 50) torch.Size([512, 3, 32, 32]) torch.float32
('DEIS', 100) torch.Size([512, 3, 32, 32]) torch.float32
('DPM++', 50) torch.Size([512, 3, 32, 32]) torch.float32
('DPM++', 100) torch.Size([512, 3, 32, 32]) torch.float32
('UniPC', 50) torch.Size([512, 3, 32, 32]) torch.float32
('UniPC', 100) torch.Size([512, 3, 32, 32]) torch.float32


## 4. Baseline MSE to the fixed DDIM-1000 reference

This is the **accuracy axis**.

In [16]:
baseline_rows = []

for (solver, nfe), x in solver_outputs.items():
    per_sample = ((x[VAL_END:] - teacher_test) ** 2).flatten(1).mean(dim=1)

    baseline_rows.append({
        "solver": solver,
        "NFE": nfe,
        "baseline_MSE": per_sample.mean().item(),
        "baseline_MSE_SE": (
            per_sample.std(unbiased=True).item() / np.sqrt(len(per_sample))
        ),
    })

df_baseline = pd.DataFrame(baseline_rows).sort_values(["NFE", "baseline_MSE"])
display(df_baseline)

df_baseline.to_csv(
    OUT_DIR / "equal_nfe_baseline_mse.csv",
    index=False
)

,solver,NFE,baseline_MSE,baseline_MSE_SE
0,DDIM,50,0.198522,0.020865
6,UniPC,50,0.223398,0.023425
2,DEIS,50,0.223748,0.023428
4,DPM++,50,0.224147,0.023478
1,DDIM,100,0.180091,0.019552
3,DEIS,100,0.222732,0.023354
7,UniPC,100,0.222745,0.023366
5,DPM++,100,0.222964,0.023382


## 5. Correction model

Same local CNN family used in the main RAC experiment.

In [18]:
class CorrectionCNN(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, width, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(width, width, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(width, width, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(width, 3, 3, padding=1),
        )

    def forward(self, x):
        return self.net(x)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def train_corrector(
    residual,
    width,
    seed=0,
    max_epochs=1000,
    batch_size=32,
    lr=1e-3,
    patience=40,
    min_delta=1e-5,
):
    set_seed(seed)

    r_train = residual[:TRAIN_END]
    r_val = residual[TRAIN_END:VAL_END]
    r_test = residual[VAL_END:]

    train_rms = torch.sqrt((r_train ** 2).mean()).item()

    y_train = r_train / train_rms

    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        TensorDataset(z_train, y_train),
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
    )

    model = CorrectionCNN(width).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    best_epoch = None
    bad_epochs = 0

    for epoch in range(1, max_epochs + 1):
        model.train()

        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            pred_val = model(z_val.to(device)).cpu() * train_rms

        val_mse = ((pred_val - r_val) ** 2).mean().item()
        val_energy = (r_val ** 2).mean().item()
        val_nmse = val_mse / val_energy

        if val_nmse < best_val - min_delta:
            best_val = val_nmse
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        pred_test = model(z_test.to(device)).cpu() * train_rms

    test_mse = ((pred_test - r_test) ** 2).mean().item()
    test_energy = (r_test ** 2).mean().item()
    test_nmse = test_mse / test_energy

    return {
        "width": width,
        "params": count_params(model),
        "seed": seed,
        "best_epoch": best_epoch,
        "val_nmse": best_val,
        "test_nmse": test_nmse,
    }

## 6. Equal-NFE correction experiment

The headline paper comparison uses width 8.  
We also include widths 1 and 4 as a compact robustness check.

This is cheap relative to diffusion generation: only small CNNs are trained.

In [19]:
WIDTHS = [8]
SEEDS = [0, 1, 2]

rows = []

for (solver, nfe), baseline in solver_outputs.items():
    residual = teacher - baseline

    for width in WIDTHS:
        for seed in SEEDS:
            print(
                f"solver={solver:5s} | NFE={nfe:3d} | "
                f"width={width:2d} | seed={seed}"
            )

            out = train_corrector(
                residual=residual,
                width=width,
                seed=seed,
            )

            rows.append({
                "solver": solver,
                "NFE": nfe,
                **out,
            })

df_corr_raw = pd.DataFrame(rows)
display(df_corr_raw.head())

df_corr_raw.to_csv(
    OUT_DIR / "equal_nfe_corrector_raw.csv",
    index=False
)

solver=DDIM  | NFE= 50 | width= 8 | seed=0
solver=DDIM  | NFE= 50 | width= 8 | seed=1
solver=DDIM  | NFE= 50 | width= 8 | seed=2
solver=DDIM  | NFE=100 | width= 8 | seed=0
solver=DDIM  | NFE=100 | width= 8 | seed=1
solver=DDIM  | NFE=100 | width= 8 | seed=2
solver=DEIS  | NFE= 50 | width= 8 | seed=0
solver=DEIS  | NFE= 50 | width= 8 | seed=1
solver=DEIS  | NFE= 50 | width= 8 | seed=2
solver=DEIS  | NFE=100 | width= 8 | seed=0
solver=DEIS  | NFE=100 | width= 8 | seed=1
solver=DEIS  | NFE=100 | width= 8 | seed=2
solver=DPM++ | NFE= 50 | width= 8 | seed=0
solver=DPM++ | NFE= 50 | width= 8 | seed=1
solver=DPM++ | NFE= 50 | width= 8 | seed=2
solver=DPM++ | NFE=100 | width= 8 | seed=0
solver=DPM++ | NFE=100 | width= 8 | seed=1
solver=DPM++ | NFE=100 | width= 8 | seed=2
solver=UniPC | NFE= 50 | width= 8 | seed=0
solver=UniPC | NFE= 50 | width= 8 | seed=1
solver=UniPC | NFE= 50 | width= 8 | seed=2
solver=UniPC | NFE=100 | width= 8 | seed=0
solver=UniPC | NFE=100 | width= 8 | seed=1
solver=UniP

,solver,NFE,width,params,seed,best_epoch,val_nmse,test_nmse
0,DDIM,50,8,1611,0,263,0.314607,0.319766
1,DDIM,50,8,1611,1,218,0.315499,0.319862
2,DDIM,50,8,1611,2,286,0.314048,0.321226
3,DDIM,100,8,1611,0,306,0.330780,0.334773
4,DDIM,100,8,1611,1,170,0.332147,0.335559


## 7. Aggregate seeds

In [20]:
df_corr = (
    df_corr_raw
    .groupby(["solver", "NFE", "width", "params"], as_index=False)
    .agg(
        residual_NMSE_mean=("test_nmse", "mean"),
        residual_NMSE_std=("test_nmse", "std"),
        seeds=("seed", "count"),
    )
)

df_equal = df_corr.merge(
    df_baseline[["solver", "NFE", "baseline_MSE"]],
    on=["solver", "NFE"],
    how="left",
)

df_equal = df_equal.sort_values(
    ["NFE", "width", "baseline_MSE"]
)

display(df_equal)

df_equal.to_csv(
    OUT_DIR / "equal_nfe_summary.csv",
    index=False\
)

,solver,NFE,width,params,residual_NMSE_mean,residual_NMSE_std,seeds,baseline_MSE
0,DDIM,50,8,1611,0.320285,0.000817,3,0.198522
6,UniPC,50,8,1611,0.318280,0.000933,3,0.223398
2,DEIS,50,8,1611,0.317274,0.000483,3,0.223748
4,DPM++,50,8,1611,0.317743,0.000946,3,0.224147
1,DDIM,100,8,1611,0.335540,0.000757,3,0.180091
3,DEIS,100,8,1611,0.318320,0.001150,3,0.222732
7,UniPC,100,8,1611,0.318166,0.000825,3,0.222745
5,DPM++,100,8,1611,0.318006,0.000829,3,0.222964


## 8. Automatically detect accuracy–difficulty inversions

For the same NFE and correction width, an inversion is a pair where:

\[
\mathrm{MSE}(A) < \mathrm{MSE}(B)
\]

but

\[
\mathrm{NMSE}_{res}(A) > \mathrm{NMSE}_{res}(B).
\]

The code below derives this directly from the observed results.

In [21]:
inversions = []

for nfe in NFES:
    for width in WIDTHS:
        sub = df_equal[
            (df_equal["NFE"] == nfe) &
            (df_equal["width"] == width)
        ].reset_index(drop=True)

        for i in range(len(sub)):
            for j in range(len(sub)):
                if i == j:
                    continue

                a = sub.iloc[i]
                b = sub.iloc[j]

                if (
                    a["baseline_MSE"] < b["baseline_MSE"]
                    and
                    a["residual_NMSE_mean"] > b["residual_NMSE_mean"]
                ):
                    inversions.append({
                        "NFE": nfe,
                        "width": width,
                        "more_accurate_solver": a["solver"],
                        "less_accurate_solver": b["solver"],
                        "MSE_more_accurate": a["baseline_MSE"],
                        "MSE_less_accurate": b["baseline_MSE"],
                        "NMSE_more_accurate": a["residual_NMSE_mean"],
                        "NMSE_less_accurate": b["residual_NMSE_mean"],
                        "NMSE_gap": (
                            a["residual_NMSE_mean"]
                            - b["residual_NMSE_mean"]
                        ),
                    })

df_inv = pd.DataFrame(inversions)

if len(df_inv):
    df_inv = (
        df_inv
        .sort_values(
            ["NFE", "width", "NMSE_gap"],
            ascending=[True, True, False],
        )
        .drop_duplicates()
    )

display(df_inv)

df_inv.to_csv(
    OUT_DIR / "equal_nfe_inversions.csv",
    index=False
)

,NFE,width,more_accurate_solver,less_accurate_solver,MSE_more_accurate,MSE_less_accurate,NMSE_more_accurate,NMSE_less_accurate,NMSE_gap
1,50,8,DDIM,DEIS,0.198522,0.223748,0.320285,0.317274,0.003011
2,50,8,DDIM,DPM++,0.198522,0.224147,0.320285,0.317743,0.002542
0,50,8,DDIM,UniPC,0.198522,0.223398,0.320285,0.318280,0.002005
3,50,8,UniPC,DEIS,0.223398,0.223748,0.318280,0.317274,0.001006
4,50,8,UniPC,DPM++,0.223398,0.224147,0.318280,0.317743,0.000537
7,100,8,DDIM,DPM++,0.180091,0.222964,0.335540,0.318006,0.017533
6,100,8,DDIM,UniPC,0.180091,0.222745,0.335540,0.318166,0.017373
5,100,8,DDIM,DEIS,0.180091,0.222732,0.335540,0.318320,0.017219
9,100,8,DEIS,DPM++,0.222732,0.222964,0.318320,0.318006,0.000314
10,100,8,UniPC,DPM++,0.222745,0.222964,0.318166,0.318006,0.000160


## 9. Paper-ready width-8 table

This table puts **baseline accuracy and residual difficulty side by side** so the
central inversion is visible without interpretation.

In [22]:
paper_table = (
    df_equal[df_equal["width"] == 8]
    [["NFE", "solver", "baseline_MSE",
      "residual_NMSE_mean", "residual_NMSE_std", "params"]]
    .sort_values(["NFE", "baseline_MSE"])
)

display(paper_table)

paper_table.to_csv(
    OUT_DIR / "equal_nfe_paper_table.csv",
    index=False
)

latex = paper_table.to_latex(
    index=False,
    float_format=lambda x: f"{x:.6f}",
)

with open(OUT_DIR / "equal_nfe_paper_table.tex", "w") as f:
    f.write(latex)

print(latex)

,NFE,solver,baseline_MSE,residual_NMSE_mean,residual_NMSE_std,params
0,50,DDIM,0.198522,0.320285,0.000817,1611
6,50,UniPC,0.223398,0.318280,0.000933,1611
2,50,DEIS,0.223748,0.317274,0.000483,1611
4,50,DPM++,0.224147,0.317743,0.000946,1611
1,100,DDIM,0.180091,0.335540,0.000757,1611
3,100,DEIS,0.222732,0.318320,0.001150,1611
7,100,UniPC,0.222745,0.318166,0.000825,1611
5,100,DPM++,0.222964,0.318006,0.000829,1611


\begin{tabular}{rlrrrr}
\toprule
NFE & solver & baseline_MSE & residual_NMSE_mean & residual_NMSE_std & params \\
\midrule
50 & DDIM & 0.198522 & 0.320285 & 0.000817 & 1611 \\
50 & UniPC & 0.223398 & 0.318280 & 0.000933 & 1611 \\
50 & DEIS & 0.223748 & 0.317274 & 0.000483 & 1611 \\
50 & DPM++ & 0.224147 & 0.317743 & 0.000946 & 1611 \\
100 & DDIM & 0.180091 & 0.335540 & 0.000757 & 1611 \\
100 & DEIS & 0.222732 & 0.318320 & 0.001150 & 1611 \\
100 & UniPC & 0.222745 & 0.318166 & 0.000825 & 1611 \\
100 & DPM++ & 0.222964 & 0.318006 & 0.000829 & 1611 \\
\bottomrule
\end{tabular}



## 10. Paper-ready figure

In [23]:
from pathlib import Path
import pandas as pd

# Folder created by Notebook 2
OUT_DIR = Path("./equal_nfe_validation")

# Load already-computed results
df_corr_raw = pd.read_csv(
    OUT_DIR / "equal_nfe_corrector_raw.csv"
)

df_baseline = pd.read_csv(
    OUT_DIR / "equal_nfe_baseline_mse.csv"
)

# Aggregate the 3 seeds
df_corr = (
    df_corr_raw
    .groupby(
        ["solver", "NFE", "width", "params"],
        as_index=False
    )
    .agg(
        residual_NMSE_mean=("test_nmse", "mean"),
        residual_NMSE_std=("test_nmse", "std"),
        seeds=("seed", "count"),
    )
)

# Join baseline accuracy with residual difficulty
df_equal = df_corr.merge(
    df_baseline[
        ["solver", "NFE", "baseline_MSE"]
    ],
    on=["solver", "NFE"],
    how="left"
)

df_equal = df_equal.sort_values(
    ["NFE", "width", "baseline_MSE"]
)

print("df_equal created successfully:")
display(df_equal)

df_equal created successfully:


,solver,NFE,width,params,residual_NMSE_mean,residual_NMSE_std,seeds,baseline_MSE
0,DDIM,50,8,1611,0.320285,0.000817,3,0.198522
6,UniPC,50,8,1611,0.318280,0.000933,3,0.223398
2,DEIS,50,8,1611,0.317274,0.000483,3,0.223748
4,DPM++,50,8,1611,0.317743,0.000946,3,0.224147
1,DDIM,100,8,1611,0.335540,0.000757,3,0.180091
3,DEIS,100,8,1611,0.318320,0.001150,3,0.222732
7,UniPC,100,8,1611,0.318166,0.000825,3,0.222745
5,DPM++,100,8,1611,0.318006,0.000829,3,0.222964


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Robust equal-NFE plot
# -----------------------------

required_cols = {
    "width",
    "NFE",
    "solver",
    "baseline_MSE",
    "residual_NMSE_mean",
}

missing = required_cols - set(df_equal.columns)

if missing:
    raise ValueError(
        f"Missing columns in df_equal: {missing}\n"
        f"Available columns: {list(df_equal.columns)}"
    )

plot_df = df_equal[df_equal["width"] == 8].copy()

if plot_df.empty:
    raise ValueError(
        "No rows found for width=8.\n"
        "Check df_equal['width'].unique()."
    )

print("Rows used for plot:")
display(
    plot_df[
        ["solver", "NFE", "baseline_MSE", "residual_NMSE_mean"]
    ].sort_values(["NFE", "baseline_MSE"])
)

# Infer NFE values directly from the actual results
nfe_values = sorted(plot_df["NFE"].unique())

marker_list = ["o", "s", "^", "D", "P", "X"]

fig, ax = plt.subplots(figsize=(7.0, 5.0))

for i, nfe in enumerate(nfe_values):

    sub = plot_df[plot_df["NFE"] == nfe]

    marker = marker_list[i % len(marker_list)]

    ax.scatter(
        sub["baseline_MSE"],
        sub["residual_NMSE_mean"],
        s=80,
        marker=marker,
        label=f"{nfe} NFE",
    )

    for _, row in sub.iterrows():
        ax.annotate(
            str(row["solver"]),
            (
                row["baseline_MSE"],
                row["residual_NMSE_mean"]
            ),
            xytext=(6, 5),
            textcoords="offset points",
            fontsize=9,
        )

ax.set_xlabel(
    "Baseline MSE to DDIM-1000 reference\n"
    "(lower = more accurate)"
)

ax.set_ylabel(
    "Held-out residual NMSE, width 8\n"
    "(lower = easier to correct)"
)

ax.set_title(
    "Equal-NFE Accuracy–Difficulty Inversions"
)

ax.grid(True, alpha=0.25)
ax.legend(frameon=False)

fig.tight_layout()

png_path = OUT_DIR / "equal_nfe_inversions.png"
pdf_path = OUT_DIR / "equal_nfe_inversions.pdf"

fig.savefig(
    png_path,
    dpi=220,
    bbox_inches="tight"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight"
)

plt.show()

print("Saved:")
print(png_path.resolve())
print(pdf_path.resolve())

Rows used for plot:


,solver,NFE,baseline_MSE,residual_NMSE_mean
0,DDIM,50,0.198522,0.320285
6,UniPC,50,0.223398,0.318280
2,DEIS,50,0.223748,0.317274
4,DPM++,50,0.224147,0.317743
1,DDIM,100,0.180091,0.335540
3,DEIS,100,0.222732,0.318320
7,UniPC,100,0.222745,0.318166
5,DPM++,100,0.222964,0.318006


## 11. Reproducibility record

In [1]:
SEED = 20260927

In [10]:
manifest = {
    "seed_base": SEED,
    "train_size": N_TRAIN,
    "val_size": N_VAL,
    "test_size": len(z_test),
    "reference": "DDIM-1000",
    "nfe_values": NFES,
    "solvers": list(ALIASES.keys()),
    "widths": WIDTHS,
    "seeds": SEEDS,
    "optimizer": "Adam",
    "learning_rate": 1e-3,
    "batch_size": 32,
    "max_epochs": 1000,
    "patience": 40,
    "normalization": "train residual RMS",
    "device": device,
    "pytorch": torch.__version__,
    "input_solver_files": {
        f"{solver}_{nfe}": str(path)
        for (solver, nfe), path in solver_files.items()
    },
}

with open(OUT_DIR / "equal_nfe_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Saved everything to:", OUT_DIR.resolve())

Saved everything to: D:\PhD EMP\Paper Writting\STEP_COMPLEXITY\equal_nfe_validation


# Interpretation rule

Do **not** write any solver ordering or numerical inversion in the manuscript until
this notebook has run successfully.

After execution, the paper may report only the rows present in
`equal_nfe_inversions.csv` and the numerical values in
`equal_nfe_paper_table.csv`.